# Lesson 4.4 — SSM OpsCenter, Parameter Store & Patch Manager

**Level: Intermediate** | **Exam:** SOA-C03 Domain 3, Skills 3.1.6, 3.2.1; Domain 1, Skill 1.2.3

## Learning Objectives
- Use SSM Patch Manager to automate OS patching with approval workflows
- Manage OpsCenter OpsItems for operational issue tracking
- Store configuration at scale using Parameter Store with hierarchies and policies
- Configure SSM Maintenance Windows for zero-disruption patching

_Reference: [Patch Manager](https://docs.aws.amazon.com/systems-manager/latest/userguide/systems-manager-patch.html) | [OpsCenter](https://docs.aws.amazon.com/systems-manager/latest/userguide/OpsCenter.html)_

## SSM Patch Manager

Patch Manager automates patching your EC2 fleet — no manual `yum update` per instance.

### Patch Baseline
Defines **which patches** are approved for installation:
- AWS-maintained baselines: `AWS-AmazonLinux2DefaultPatchBaseline`, `AWS-WindowsDefaultPatchBaseline`, etc.
- Custom baselines: Choose severity levels (Critical, Important, etc.) and products
- **Approval delay**: e.g., auto-approve security patches after 7 days (gives time to test)

### Patch Groups
- Tag instances with `Key=Patch Group, Value=<group-name>`
- Associate a patch baseline with a patch group
- Example: `Patch Group=WebServers` → Web server baseline (more conservative)

### Maintenance Windows
- Define a **time window** for patching (e.g., Tuesday 2–4 AM)
- Associate tasks (Run Command with `AWS-RunPatchBaseline` document)
- Set `CutoffMinutes` to stop starting new tasks before window ends

## SSM OpsCenter

OpsCenter aggregates **OpsItems** — operational issues from CloudWatch Alarms, EventBridge, etc.:
- Manually create OpsItems for incidents
- Auto-create from CloudWatch Alarms (using EventBridge rules)
- Attach runbooks (SSM Automation) directly to OpsItems for one-click remediation

## Parameter Store Policies

Advanced tier parameters support **policies**:
```json
[
  {
    "Type": "Expiration",
    "Version": "1.0",
    "Attributes": {
      "Timestamp": "2025-12-31T23:59:59.000Z"
    }
  },
  {
    "Type": "ExpirationNotification",
    "Version": "1.0",
    "Attributes": {
      "Before": "7",
      "Unit": "Days"
    }
  }
]
```
Use policies for rotating credentials — parameter expires, triggering a Lambda to rotate it.

In [ ]:
import boto3, json

ssm = boto3.client('ssm', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Explore AWS-managed patch baselines
paginator = ssm.get_paginator('describe_patch_baselines')

print('AWS-Managed Patch Baselines:')
for page in paginator.paginate(Filters=[{'Key': 'OWNER', 'Values': ['AWS']}]):
    for baseline in page['BaselineIdentities']:
        if 'Amazon' in baseline['BaselineName'] or 'Windows' in baseline['BaselineName']:
            print(f"  {baseline['OperatingSystem']:20} {baseline['BaselineName']}")

In [ ]:
# Create a Maintenance Window for Tuesday 2 AM UTC patching

try:
    window = ssm.create_maintenance_window(
        Name='cloudops-tuesday-patching',
        Description='Automated patching every Tuesday 2-4 AM UTC',
        Schedule='cron(0 2 ? * TUE *)',    # Every Tuesday at 2 AM UTC
        Duration=2,                          # 2-hour window
        Cutoff=1,                            # Stop starting new tasks 1 hour before end
        AllowUnassociatedTargets=False
    )
    window_id = window['WindowId']
    print(f'Created Maintenance Window: {window_id}')
except ssm.exceptions.IdempotentParameterMismatch:
    windows = ssm.describe_maintenance_windows(Filters=[{'Key': 'Name', 'Values': ['cloudops-tuesday-patching']}])
    window_id = windows['WindowIdentities'][0]['WindowId']
    print(f'Maintenance Window exists: {window_id}')
except Exception as e:
    print(f'Maintenance window: {e}')

print('\nPatch Compliance Summary Query:')
print('To get patch compliance across all managed instances:')
print('ssm.describe_instance_patch_states_for_patch_group(PatchGroup="WebServers")')

In [ ]:
# Create an OpsItem manually (simulating an incident)

try:
    opsitem = ssm.create_ops_item(
        Title='High CPU on production web server',
        Description='CPU utilization exceeded 90% for 15 minutes on web-prod-001',
        Source='cloudwatch-alarm',
        Priority=2,    # 1=critical, 2=high, 3=medium, 4=low
        OperationalData={
            'InstanceId': {'Value': 'i-1234567890abcdef0', 'Type': 'SearchableString'},
            'AffectedEnvironment': {'Value': 'production', 'Type': 'SearchableString'},
            'Runbook': {'Value': 'https://wiki.company.com/runbooks/high-cpu', 'Type': 'String'}
        },
        Notifications=[
            {'Arn': f'arn:aws:sns:us-east-1:{account_id}:CloudOpsAlerts'}
        ],
        Tags=[{'Key': 'Environment', 'Value': 'prod'}]
    )
    opsitem_id = opsitem['OpsItemId']
    print(f'Created OpsItem: {opsitem_id}')
except Exception as e:
    print(f'OpsItem: {e}')

# List open OpsItems
try:
    items = ssm.describe_ops_items(
        OpsItemFilters=[{'Key': 'Status', 'Values': ['Open'], 'Operator': 'Equal'}]
    )['OpsItemSummaries']
    print(f'\nOpen OpsItems ({len(items)}):')
    for item in items[:5]:
        print(f'  [{item["Priority"]}] {item["OpsItemId"]:15} {item["Title"][:60]}')
except Exception as e:
    print(f'OpsItems list: {e}')

## Exercise

In [ ]:
# TODO: Write a Python function that generates a compliance report for SSM
# The function should:
# 1. List all managed instances (ssm.describe_instance_information())
# 2. For each instance, check patch compliance (ssm.describe_instance_patch_states())
# 3. Return a dict: {instance_id: {'compliant': bool, 'missing_patches': int, 'failed_patches': int}}

import boto3

def get_patch_compliance_report() -> dict:
    ssm = boto3.client('ssm', region_name='us-east-1')
    report = {}
    
    # YOUR CODE HERE
    
    return report

result = get_patch_compliance_report()
if result:
    print(f'Patch Compliance Report ({len(result)} instances):')
    for instance_id, status in result.items():
        compliance = 'COMPLIANT' if status['compliant'] else 'NON-COMPLIANT'
        print(f'  {instance_id}: {compliance} (missing: {status["missing_patches"]}, failed: {status["failed_patches"]})')
else:
    print('No managed instances found or function not implemented yet.')

In [ ]:
# Cleanup
import boto3
ssm = boto3.client('ssm', region_name='us-east-1')

try:
    windows = ssm.describe_maintenance_windows(Filters=[{'Key': 'Name', 'Values': ['cloudops-tuesday-patching']}])
    for w in windows['WindowIdentities']:
        ssm.delete_maintenance_window(WindowId=w['WindowId'])
        print(f'Deleted maintenance window: {w["WindowId"]}')
except Exception as e:
    print(f'Cleanup: {e}')

## Summary
- **Patch Manager**: Automates OS patching using patch baselines, patch groups, and maintenance windows
- **Patch groups** (tag `Patch Group`) link instances to specific baselines
- **OpsCenter**: Central place to track, manage, and remediate operational issues (OpsItems)
- **Maintenance Windows**: Schedule operational tasks (patching, backups) during approved time windows
- Advanced Parameter Store parameters support **expiry policies** for credential rotation

---
**Next:** [Lesson 4.5 — Event-Driven Automation: Lambda, S3 Events & EventBridge](lesson-4.5-event-driven-automation.ipynb)